# 4.7 Cluster Splitter

**Usage — restart kernel and run all cells top-to-bottom:**

1. Only clusters marked `✂ Split` in 4.6 appear here.
2. Use the **Cluster** dropdown or **◀ Prev / Next ▶** to navigate.
3. Choose the **split axis** (X or Y in cluster-centred coords) and drag the **threshold slider** to position the split line. The preview updates when you release the slider.
4. Set the **Part A** and **Part B** labels, then click **✂ Apply Split**.
5. The original cluster is retired and two new clusters are added to `inventory.csv`. They show up as normal labeled clusters in 4.6.

> The split is axis-aligned (vertical or horizontal line through the top view). If the two objects share a diagonal boundary, choose the axis that best separates them.

In [ ]:
import sys, io
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from shapely.geometry import MultiPoint

from config import CLUSTERS_DIR

In [ ]:
INV_PATH = CLUSTERS_DIR / 'inventory.csv'
inv = pd.read_csv(INV_PATH)

for col, default in [('final_label', None), ('label_source_final', None),
                      ('needs_review', False), ('needs_split', False), ('split_done', False)]:
    if col not in inv.columns:
        inv[col] = default

inv['label_source_final'] = inv['label_source_final'].astype(object)
inv['final_label']        = inv['final_label'].astype(object)

def _queue():
    return inv[
        inv['needs_split'].fillna(False).astype(bool) &
        ~inv['split_done'].fillna(False).astype(bool)
    ]

print(f'Clusters queued for splitting: {len(_queue())}')

In [ ]:
LABEL_NAMES = {
    0: 'Unknown', 1: 'Road', 9: 'Ground', 10: 'Building', 11: 'Facade',
    15: 'Bus/tram shelter',
    30: 'Tree', 39: 'Green (other)', 40: 'Car', 44: 'Bicycle', 50: 'Person',
    60: 'Street Light', 61: 'Traffic Light', 62: 'Traffic Sign',
    65: 'Bollard', 67: 'Stop Pole', 80: 'City Bench',
    81: 'Rubbish Bin', 83: 'Large Container', 85: 'Parking Meter',
    88: 'Bicycle Rack', 99: 'Noise / False Positive',
}
LABEL_OPTIONS = [(f"{v}  ({k})", k) for k, v in sorted(LABEL_NAMES.items())]

In [ ]:
# ── rendering helpers ──────────────────────────────────────────────────────────
def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')

def _fig_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='#1a1a1a', bbox_inches='tight')
    buf.seek(0)
    return buf.read()


def _split_mask(xyz, hag, axis, threshold):
    """Return boolean mask for Part A (<= threshold) on the chosen axis."""
    if axis == 2:
        return hag <= threshold   # Z split uses height above ground
    return xyz[:, axis] <= threshold


def render_split_preview(row, axis, threshold):
    """Top view + side view coloured by part A/B with split line.
    Returns (png_bytes, n_pts_a, n_pts_b)."""
    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        fig = Figure(figsize=(10, 4), facecolor='#1a1a1a')
        FigureCanvasAgg(fig)
        ax = fig.add_subplot(111); _sa(ax)
        ax.text(0.5, 0.5, f'Cannot load NPZ:\n{e}', color='#cc4444',
                ha='center', va='center', transform=ax.transAxes)
        return _fig_bytes(fig), 0, 0

    xyz  = npz['xyz_centered']
    rgb  = np.clip(npz['rgb_norm'], 0, 1)
    hag  = npz['height_ag']
    pt_sz = max(1, min(10, 3000 // max(len(xyz), 1)))

    mask_a = _split_mask(xyz, hag, axis, threshold)
    mask_b = ~mask_a
    n_a, n_b = int(mask_a.sum()), int(mask_b.sum())

    fig = Figure(figsize=(10, 4), facecolor='#1a1a1a')
    FigureCanvasAgg(fig)
    ax_t = fig.add_subplot(1, 2, 1)
    ax_s = fig.add_subplot(1, 2, 2)
    _sa(ax_t); _sa(ax_s)

    # Top view (XY) — coloured by part for X/Y splits; scan colours for Z split
    if axis == 2:
        ax_t.scatter(xyz[:, 0], xyz[:, 1], c=rgb, s=pt_sz, linewidths=0)
        ax_t.set_title('Top view (XY) — Z split', color='#aaa', fontsize=8)
    else:
        if mask_a.any():
            ax_t.scatter(xyz[mask_a, 0], xyz[mask_a, 1], c='#4488ff', s=pt_sz,
                         alpha=0.8, linewidths=0, label=f'Part A ({n_a} pts)')
        if mask_b.any():
            ax_t.scatter(xyz[mask_b, 0], xyz[mask_b, 1], c='#ff8844', s=pt_sz,
                         alpha=0.8, linewidths=0, label=f'Part B ({n_b} pts)')
        if axis == 0:
            ax_t.axvline(threshold, color='white', linewidth=1.5, linestyle='--', zorder=10)
        else:
            ax_t.axhline(threshold, color='white', linewidth=1.5, linestyle='--', zorder=10)
        ax_t.set_title(f'Top view · {"X" if axis==0 else "Y"} split at {threshold:.2f} m',
                       color='#aaa', fontsize=8)
        ax_t.legend(facecolor='#1e1e1e', labelcolor='white', edgecolor='#444',
                    fontsize=7, loc='upper right')
    ax_t.set_aspect('equal')
    ax_t.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_t.set_ylabel('ΔY (m)', color='grey', fontsize=7)

    # Side view (XZ with height_ag on Y) — split line for Z; scan colours otherwise
    if axis == 2:
        if mask_a.any():
            ax_s.scatter(xyz[mask_a, 0], hag[mask_a], c='#4488ff', s=pt_sz,
                         alpha=0.8, linewidths=0, label=f'Part A ({n_a} pts)')
        if mask_b.any():
            ax_s.scatter(xyz[mask_b, 0], hag[mask_b], c='#ff8844', s=pt_sz,
                         alpha=0.8, linewidths=0, label=f'Part B ({n_b} pts)')
        ax_s.axhline(threshold, color='white', linewidth=1.5, linestyle='--', zorder=10)
        ax_s.set_title(f'Side view · Z split at {threshold:.2f} m above ground',
                       color='#aaa', fontsize=8)
        ax_s.legend(facecolor='#1e1e1e', labelcolor='white', edgecolor='#444',
                    fontsize=7, loc='upper right')
    else:
        ax_s.scatter(xyz[:, 0], hag, c=rgb, s=pt_sz, linewidths=0)
        if axis == 0:
            ax_s.axvline(threshold, color='white', linewidth=1.5, linestyle='--', zorder=10)
        ax_s.set_title('Side view (height above ground)', color='#aaa', fontsize=8)
    ax_s.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_s.set_ylabel('height ag (m)', color='grey', fontsize=7)

    fig.tight_layout()
    return _fig_bytes(fig), n_a, n_b


def _slider_range(row, axis):
    """Return (min, max, step, initial) for the split threshold slider."""
    try:
        npz = np.load(row['npz_path'])
        vals = npz['height_ag'] if axis == 2 else npz['xyz_centered'][:, axis]
    except Exception:
        return -5.0, 5.0, 0.05, 0.0
    lo   = float(vals.min())
    hi   = float(vals.max())
    mid  = round((lo + hi) / 2, 2)
    step = max(0.01, round((hi - lo) / 200, 2))
    return round(lo, 2), round(hi, 2), step, mid


def _save_split(inv_idx, axis, threshold, label_a, label_b):
    """Create two sub-cluster NPZs, append rows to inventory, retire original."""
    global inv
    row = inv.loc[inv_idx]

    npz_data = np.load(row['npz_path'])
    xyz = npz_data['xyz_centered']
    rgb = npz_data['rgb_norm']
    hag = npz_data['height_ag']

    orig_cx  = float(row['centroid_x'])
    orig_cy  = float(row['centroid_y'])
    tilecode = str(row['tilecode'])
    tile_dir = CLUSTERS_DIR / tilecode

    mask_a = _split_mask(xyz, hag, axis, threshold)
    mask_b = ~mask_a

    next_idx = int(inv[inv['tilecode'] == tilecode]['cluster_idx'].max()) + 1

    new_rows = []
    for i, (mask, label) in enumerate([(mask_a, label_a), (mask_b, label_b)]):
        pts_xyz_c = xyz[mask].astype(np.float64)
        abs_x = pts_xyz_c[:, 0] + orig_cx
        abs_y = pts_xyz_c[:, 1] + orig_cy
        cx = float(abs_x.mean())
        cy = float(abs_y.mean())

        xyz_c = pts_xyz_c.copy().astype(np.float32)
        xyz_c[:, 0] = (abs_x - cx).astype(np.float32)
        xyz_c[:, 1] = (abs_y - cy).astype(np.float32)

        pts_rgb = rgb[mask]
        pts_h   = hag[mask]

        try:
            area = float(MultiPoint(
                np.column_stack([abs_x, abs_y])
            ).convex_hull.area)
        except Exception:
            area = 0.0

        cluster_idx = next_idx + i
        npz_path = tile_dir / f'cluster_{cluster_idx:04d}.npz'
        np.savez_compressed(
            npz_path,
            xyz_centered = xyz_c,
            rgb_norm     = pts_rgb.astype(np.float32),
            height_ag    = pts_h.astype(np.float32),
            centroid_xy  = np.array([cx, cy], dtype=np.float64),
            label        = np.int32(label),
            label_frac   = np.float32(0.0),
            label_source = np.bytes_('manual_split'),
            n_raw_pts    = np.int32(int(mask.sum())),
            area_m2      = np.float32(area),
            tilecode     = np.bytes_(tilecode),
            cluster_idx  = np.int32(cluster_idx),
        )
        lname = LABEL_NAMES.get(label, str(label))
        new_rows.append({
            'tilecode':           tilecode,
            'cluster_idx':        cluster_idx,
            'npz_path':           str(npz_path),
            'label':              label,
            'label_frac':         0.0,
            'label_source':       'manual_split',
            'final_label':        label,
            'label_source_final': 'manual_split',
            'needs_review':       False,
            'needs_split':        False,
            'split_done':         False,
            'n_raw_pts':          int(mask.sum()),
            'area_m2':            round(area, 3),
            'centroid_x':         round(cx, 2),
            'centroid_y':         round(cy, 2),
            'timestamp':          None,
        })
        print(f'  Part {"A" if i==0 else "B"}: cluster #{cluster_idx} · {int(mask.sum())} pts · {lname}')

    inv.at[inv_idx, 'needs_split'] = False
    inv.at[inv_idx, 'split_done']  = True

    new_df = pd.DataFrame(new_rows)
    inv    = pd.concat([inv, new_df], ignore_index=True)
    inv.to_csv(INV_PATH, index=False)
    print(f'Saved. Remaining in queue: {len(_queue())}')

In [ ]:
# ── Cluster Splitter UI ────────────────────────────────────────────────────────

def _queue_options():
    q = _queue()
    opts = []
    for _, r in q.iterrows():
        lname = LABEL_NAMES.get(int(r['label']), str(r['label']))
        opts.append((f"#{int(r['cluster_idx']):>3}  {lname:<18}  {int(r['n_raw_pts']):>7,} pts",
                     r.name))
    return opts


# ── widgets ────────────────────────────────────────────────────────────────────
queue_html = widgets.HTML(value='')

cluster_dd = widgets.Dropdown(
    options=_queue_options(),
    description='Cluster:',
    layout=widgets.Layout(width='460px'),
    style={'description_width': '60px'},
)
btn_prev = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))

axis_toggle = widgets.ToggleButtons(
    options=[('Split on X axis', 0), ('Split on Y axis', 1), ('Split on Z axis', 2)],
    value=0,
    description='',
    layout=widgets.Layout(width='480px'),
)

threshold_slider = widgets.FloatSlider(
    min=-5.0, max=5.0, step=0.05, value=0.0,
    description='Threshold (m):',
    continuous_update=False,
    layout=widgets.Layout(width='480px'),
    style={'description_width': '110px'},
)

preview_img = widgets.Image(value=b'', format='png',
                             layout=widgets.Layout(width='820px'))
pts_html = widgets.HTML(value='')

label_a_dd = widgets.Dropdown(
    options=LABEL_OPTIONS, value=0,
    description='Part A label:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '90px'},
)
label_b_dd = widgets.Dropdown(
    options=LABEL_OPTIONS, value=0,
    description='Part B label:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '90px'},
)

btn_apply = widgets.Button(
    description='✂ Apply Split',
    button_style='danger',
    layout=widgets.Layout(width='160px'),
    tooltip='Save the two sub-clusters and retire the original',
)
status_html = widgets.HTML(value='')


# ── refresh ────────────────────────────────────────────────────────────────────
def _update_queue_html():
    n = len(_queue())
    queue_html.value = (f'<span style="color:#aaa;font-size:12px"><b>{n}</b> cluster(s) queued for splitting</span>')

def _update_slider(row, axis):
    lo, hi, step, mid = _slider_range(row, axis)
    threshold_slider.unobserve_all()
    threshold_slider.min   = lo
    threshold_slider.max   = hi
    threshold_slider.step  = step
    threshold_slider.value = mid
    threshold_slider.observe(_on_threshold, names='value')

def _refresh_preview():
    idx = cluster_dd.value
    if idx is None or idx not in inv.index:
        return
    row  = inv.loc[idx]
    axis = int(axis_toggle.value)
    thr  = float(threshold_slider.value)
    png, n_a, n_b = render_split_preview(row, axis, thr)
    preview_img.value = png
    min_ok = min(n_a, n_b) >= 10
    col = '#aaffaa' if min_ok else '#ffaaaa'
    pts_html.value = (
        f'<span style="color:{col};font-size:12px">'
        f'Part A: <b>{n_a}</b> pts · Part B: <b>{n_b}</b> pts'
        + ('' if min_ok else '  ⚠ one part has < 10 pts')
        + '</span>'
    )
    btn_apply.disabled = not min_ok


# ── event handlers ─────────────────────────────────────────────────────────────
def _on_cluster(change):
    if change['name'] != 'value' or change['new'] is None:
        return
    row = inv.loc[change['new']]
    orig_lbl = int(row['label'])
    for dd in [label_a_dd, label_b_dd]:
        dd.value = orig_lbl if orig_lbl in dict(LABEL_OPTIONS).values() else 0
    _update_slider(row, int(axis_toggle.value))
    _refresh_preview()
    status_html.value = ''

def _on_axis(change):
    if change['name'] != 'value' or cluster_dd.value is None:
        return
    _update_slider(inv.loc[cluster_dd.value], int(change['new']))
    _refresh_preview()

def _on_threshold(change):
    _refresh_preview()

def _on_prev(_):
    vals = [o[1] for o in cluster_dd.options]
    if not vals: return
    i = vals.index(cluster_dd.value)
    if i > 0: cluster_dd.value = vals[i - 1]

def _on_next(_):
    vals = [o[1] for o in cluster_dd.options]
    if not vals: return
    i = vals.index(cluster_dd.value)
    if i < len(vals) - 1: cluster_dd.value = vals[i + 1]

def _on_apply(_):
    idx = cluster_dd.value
    if idx is None or idx not in inv.index:
        return
    axis  = int(axis_toggle.value)
    thr   = float(threshold_slider.value)
    lbl_a = int(label_a_dd.value)
    lbl_b = int(label_b_dd.value)
    status_html.value = '<span style="color:#ffaa44">Saving…</span>'
    _save_split(idx, axis, thr, lbl_a, lbl_b)
    status_html.value = '<span style="color:#aaffaa">✔ Saved</span>'
    opts = _queue_options()
    cluster_dd.options = opts
    _update_queue_html()
    if opts:
        cluster_dd.value = opts[0][1]
    else:
        preview_img.value = b''
        pts_html.value = ''
        status_html.value = '<span style="color:#aaffaa">✔ All clusters split.</span>'

cluster_dd.observe(_on_cluster)
axis_toggle.observe(_on_axis, names='value')
threshold_slider.observe(_on_threshold, names='value')
btn_prev.on_click(_on_prev)
btn_next.on_click(_on_next)
btn_apply.on_click(_on_apply)


# ── layout ─────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([cluster_dd, btn_prev, btn_next, queue_html],
                 layout=widgets.Layout(gap='8px', align_items='center')),
    widgets.HBox([axis_toggle, threshold_slider],
                 layout=widgets.Layout(gap='16px', align_items='center')),
    preview_img,
    pts_html,
    widgets.HBox([label_a_dd, label_b_dd],
                 layout=widgets.Layout(gap='16px')),
    widgets.HBox([btn_apply, status_html],
                 layout=widgets.Layout(gap='12px', align_items='center')),
]))


# ── initial render ─────────────────────────────────────────────────────────────
_update_queue_html()
if cluster_dd.options:
    row0 = inv.loc[cluster_dd.options[0][1]]
    orig_lbl = int(row0['label'])
    for dd in [label_a_dd, label_b_dd]:
        dd.value = orig_lbl if orig_lbl in dict(LABEL_OPTIONS).values() else 0
    _update_slider(row0, 0)
    _refresh_preview()
else:
    queue_html.value = '<span style="color:#aaffaa">No clusters queued for splitting.</span>'